In [5]:
import pandas as pd
from collections import defaultdict

In [6]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# show all columns
pd.set_option("display.max_columns", None)

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [8]:
%%R

require('tidyverse')
require('DescTools')

In [9]:


import glob

#loading multiple csv files from the file folder
csv_files = glob.glob("/Users/hazelgandhi/Desktop/tennis-regression/wta_files/*.csv")

# Step 2: Read and concatenate them
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Optional: Check the shape or preview
# print(df.shape)
# df.head()
df = df.sort_values('tourney_date')

five_set = ['Us Open','Roland Garros', 'Australian Open', 'Wimbledon']
df['five_set'] = df.tourney_name.isin(five_set)


# defining elo ratings
elo_ratings = defaultdict(lambda: 1500)

# Store Elo snapshot *before* each match
elo_snapshots = []

K = 30

def win_prob(rating_i, rating_j):
    return 1 / (1 + 10 ** ((rating_j - rating_i) / 400)) #elo formula where K=30. this is the general K value used for elo ratings

for _, match in df.iterrows():
    winner = match['winner_name']
    loser = match['loser_name']

    rating_winner = elo_ratings[winner]
    rating_loser = elo_ratings[loser]

    # Record pre-match Elo ratings
    elo_snapshots.append({
        'date': match['tourney_date'],
        'surface': match['surface'],
        'tournament': match['tourney_name'],
        'winner': winner,
        'loser': loser,
        'five_set': match['five_set'],
        'winner_elo_before': rating_winner,
        'loser_elo_before': rating_loser
    })

    # Calculate expected outcomes
    expected_win = win_prob(rating_winner, rating_loser)
    expected_loss = 1 - expected_win

    # updating ratings after every match
    elo_ratings[winner] += K * (1 - expected_win)
    elo_ratings[loser] += K * (0 - expected_loss)

# Create DataFrame
elo_df = pd.DataFrame(elo_snapshots)
elo_df

,date,surface,tournament,winner,loser,five_set,winner_elo_before,loser_elo_before
0,20220103,Hard,Adelaide 1,Elena Rybakina,Marie Bouzkova,False,1500.000000,1500.000000
1,20220103,Hard,Melbourne 2,Irina Camelia Begu,Jasmine Paolini,False,1500.000000,1500.000000
2,20220103,Hard,Melbourne 2,Aliaksandra Sasnovich,Clara Tauson,False,1500.000000,1500.000000
3,20220103,Hard,Melbourne 2,Ann Li,Kamilla Rakhimova,False,1500.000000,1500.000000
4,20220103,Hard,Melbourne 2,Daria Kasatkina,Nuria Parrizas Diaz,False,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...
8088,20241125,Clay,Buenos Aires 125,Mayar Sherif,Valeriya Strakhova,False,1531.302034,1500.000000
8089,20241125,Clay,Buenos Aires 125,Sara Bejlek,Robin Montgomery,False,1506.448439,1486.117218
8090,20241125,Clay,Buenos Aires 125,Angela Fita Boluda,Martina Capurro Taborda,False,1471.909680,1539.690753
8091,20241125,Clay,Buenos Aires 125,Katarzyna Kawa,Varvara Lepchenko,False,1499.242987,1484.791941


In [10]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Randomly assign winner to be Player A or B
winner_is_A = np.random.randint(0, 2, size=len(elo_df)) == 1

# Create transformed DataFrame
elo_transformed = pd.DataFrame({
    'date': elo_df['date'],
    'tournament': elo_df['tournament'],
    'surface' : elo_df['surface'],
    'five_set': elo_df['five_set'],
    'player_A_name': np.where(winner_is_A, elo_df['winner'], elo_df['loser']),
    'player_B_name': np.where(winner_is_A, elo_df['loser'], elo_df['winner']),
    
    'player_A_elo_before': np.where(winner_is_A, elo_df['winner_elo_before'], elo_df['loser_elo_before']),
    'player_B_elo_before': np.where(winner_is_A, elo_df['loser_elo_before'], elo_df['winner_elo_before']),
    
    'player_A_win': winner_is_A.astype(int)
})
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win
0,20220103,Adelaide 1,Hard,False,Marie Bouzkova,Elena Rybakina,1500.000000,1500.000000,0
1,20220103,Melbourne 2,Hard,False,Irina Camelia Begu,Jasmine Paolini,1500.000000,1500.000000,1
2,20220103,Melbourne 2,Hard,False,Clara Tauson,Aliaksandra Sasnovich,1500.000000,1500.000000,0
3,20220103,Melbourne 2,Hard,False,Kamilla Rakhimova,Ann Li,1500.000000,1500.000000,0
4,20220103,Melbourne 2,Hard,False,Nuria Parrizas Diaz,Daria Kasatkina,1500.000000,1500.000000,0
...,...,...,...,...,...,...,...,...,...
8088,20241125,Buenos Aires 125,Clay,False,Mayar Sherif,Valeriya Strakhova,1531.302034,1500.000000,1
8089,20241125,Buenos Aires 125,Clay,False,Sara Bejlek,Robin Montgomery,1506.448439,1486.117218,1
8090,20241125,Buenos Aires 125,Clay,False,Angela Fita Boluda,Martina Capurro Taborda,1471.909680,1539.690753,1
8091,20241125,Buenos Aires 125,Clay,False,Katarzyna Kawa,Varvara Lepchenko,1499.242987,1484.791941,1


In [11]:
elo_transformed['elo_difference'] = elo_transformed['player_A_elo_before'] - elo_transformed ['player_B_elo_before']
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference
0,20220103,Adelaide 1,Hard,False,Marie Bouzkova,Elena Rybakina,1500.000000,1500.000000,0,0.000000
1,20220103,Melbourne 2,Hard,False,Irina Camelia Begu,Jasmine Paolini,1500.000000,1500.000000,1,0.000000
2,20220103,Melbourne 2,Hard,False,Clara Tauson,Aliaksandra Sasnovich,1500.000000,1500.000000,0,0.000000
3,20220103,Melbourne 2,Hard,False,Kamilla Rakhimova,Ann Li,1500.000000,1500.000000,0,0.000000
4,20220103,Melbourne 2,Hard,False,Nuria Parrizas Diaz,Daria Kasatkina,1500.000000,1500.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...
8088,20241125,Buenos Aires 125,Clay,False,Mayar Sherif,Valeriya Strakhova,1531.302034,1500.000000,1,31.302034
8089,20241125,Buenos Aires 125,Clay,False,Sara Bejlek,Robin Montgomery,1506.448439,1486.117218,1,20.331221
8090,20241125,Buenos Aires 125,Clay,False,Angela Fita Boluda,Martina Capurro Taborda,1471.909680,1539.690753,1,-67.781073
8091,20241125,Buenos Aires 125,Clay,False,Katarzyna Kawa,Varvara Lepchenko,1499.242987,1484.791941,1,14.451046


In [12]:
surface_stats = elo_transformed.groupby(['player_A_name', 'surface'])['player_A_win'].agg(['sum', 'count']).reset_index()
surface_stats.columns = ['player', 'surface', 'wins', 'matches']
surface_stats['win_pct'] = surface_stats['wins'] / surface_stats['matches']

surface_stats

,player,surface,wins,matches,win_pct
0,Adrijana Lekaj,Clay,1,3,0.333333
1,Ajla Tomljanovic,Clay,6,10,0.600000
2,Ajla Tomljanovic,Grass,6,9,0.666667
3,Ajla Tomljanovic,Hard,13,30,0.433333
4,Akasha Urhobo,Hard,0,1,0.000000
...,...,...,...,...,...
966,Zhuoxuan Bai,Clay,0,3,0.000000
967,Zhuoxuan Bai,Grass,0,2,0.000000
968,Zhuoxuan Bai,Hard,1,3,0.333333
969,Ziva Falkner,Clay,1,1,1.000000


In [13]:
surface_wide = surface_stats.pivot(index='player', columns='surface', values='win_pct').fillna(0)
surface_wide.columns = [f"{col}_win_pct" for col in surface_wide.columns]
surface_wide.reset_index(inplace=True)
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct
0,Adrijana Lekaj,0.333333,0.000000,0.000000
1,Ajla Tomljanovic,0.600000,0.666667,0.433333
2,Akasha Urhobo,0.000000,0.000000,0.000000
3,Aldila Sutjiadi,0.000000,0.000000,0.000000
4,Aleksandra Krunic,0.500000,0.600000,0.285714
...,...,...,...,...
501,Zhaoxuan Yang,0.000000,0.000000,0.000000
502,Zhibek Kulambayeva,0.000000,0.000000,0.000000
503,Zhuoxuan Bai,0.000000,0.000000,0.333333
504,Ziva Falkner,1.000000,0.000000,0.000000


In [14]:
## Adding clusters
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = surface_wide[['Clay_win_pct', 'Grass_win_pct', 'Hard_win_pct']]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

surface_wide['surface_cluster'] = clusters
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct,surface_cluster
0,Adrijana Lekaj,0.333333,0.000000,0.000000,1
1,Ajla Tomljanovic,0.600000,0.666667,0.433333,0
2,Akasha Urhobo,0.000000,0.000000,0.000000,1
3,Aldila Sutjiadi,0.000000,0.000000,0.000000,1
4,Aleksandra Krunic,0.500000,0.600000,0.285714,0
...,...,...,...,...,...
501,Zhaoxuan Yang,0.000000,0.000000,0.000000,1
502,Zhibek Kulambayeva,0.000000,0.000000,0.000000,1
503,Zhuoxuan Bai,0.000000,0.000000,0.333333,1
504,Ziva Falkner,1.000000,0.000000,0.000000,3


In [15]:
player_clusters = surface_wide[['player', 'surface_cluster']]


In [16]:
# For player A
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_A_name', right_on='player', how='left')
elo_transformed.rename(columns={'surface_cluster': 'surface_cluster_A'}, inplace=True)
elo_transformed.drop(columns='player', inplace=True)

# For player B
player_clusters.columns = ['player', 'surface_cluster_B']
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_B_name', right_on='player', how='left')
elo_transformed.drop(columns='player', inplace=True)
elo_transformed


,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference,surface_cluster_A,surface_cluster_B
0,20220103,Adelaide 1,Hard,False,Marie Bouzkova,Elena Rybakina,1500.000000,1500.000000,0,0.000000,0.0,0.0
1,20220103,Melbourne 2,Hard,False,Irina Camelia Begu,Jasmine Paolini,1500.000000,1500.000000,1,0.000000,0.0,0.0
2,20220103,Melbourne 2,Hard,False,Clara Tauson,Aliaksandra Sasnovich,1500.000000,1500.000000,0,0.000000,3.0,2.0
3,20220103,Melbourne 2,Hard,False,Kamilla Rakhimova,Ann Li,1500.000000,1500.000000,0,0.000000,2.0,0.0
4,20220103,Melbourne 2,Hard,False,Nuria Parrizas Diaz,Daria Kasatkina,1500.000000,1500.000000,0,0.000000,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8088,20241125,Buenos Aires 125,Clay,False,Mayar Sherif,Valeriya Strakhova,1531.302034,1500.000000,1,31.302034,3.0,NaN
8089,20241125,Buenos Aires 125,Clay,False,Sara Bejlek,Robin Montgomery,1506.448439,1486.117218,1,20.331221,3.0,0.0
8090,20241125,Buenos Aires 125,Clay,False,Angela Fita Boluda,Martina Capurro Taborda,1471.909680,1539.690753,1,-67.781073,1.0,3.0
8091,20241125,Buenos Aires 125,Clay,False,Katarzyna Kawa,Varvara Lepchenko,1499.242987,1484.791941,1,14.451046,0.0,3.0


In [17]:
elo_transformed = elo_transformed.dropna()

In [18]:
%%R -i elo_transformed

logistic_new <- glm(player_A_win ~ elo_difference + elo_difference:five_set + surface:factor(surface_cluster_A) + surface:factor(surface_cluster_B), data=elo_transformed, family=binomial)
print(summary(logistic_new))
print(PseudoR2(logistic_new, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set + 
    surface:factor(surface_cluster_A) + surface:factor(surface_cluster_B), 
    family = binomial, data = elo_transformed)

Coefficients: (1 not defined because of singularities)
                                          Estimate Std. Error z value Pr(>|z|)
(Intercept)                             -0.6703508  0.1013406  -6.615 3.72e-11
elo_difference                           0.0048089  0.0002716  17.705  < 2e-16
elo_difference:five_setTRUE              0.0007738  0.0005301   1.460 0.144351
surfaceClay:factor(surface_cluster_A)0   0.6181056  0.1251695   4.938 7.89e-07
surfaceGrass:factor(surface_cluster_A)0  1.0408172  0.1366005   7.619 2.55e-14
surfaceHard:factor(surface_cluster_A)0   0.6818747  0.1051127   6.487 8.75e-11
surfaceClay:factor(surface_cluster_A)1  -1.8163768  0.2427711  -7.482 7.33e-14
surfaceGrass:factor(surface_cluster_A)1 -1.0404103  0.5018106  -2.073 0.038143
surfaceHard:factor(surface_clu

In [19]:
%%R -i elo_transformed

df <- elo_transformed %>% mutate(
    predict_proba_R = predict(logistic_new, type="response"),
    predict_R = ifelse(predict_proba_R > .5, 1,0)
) %>% arrange(elo_difference)

df %>% head()

         date     tournament surface five_set             player_A_name
6674 20240527  Roland Garros    Clay     TRUE           Leolia Jeanjean
7138 20240729 Paris Olympics    Clay    FALSE Anna Karolina Schmiedlova
6946 20240701      Wimbledon   Grass     TRUE              Petra Martic
7455 20240826        Us Open    Hard     TRUE             Ena Shibahara
6513 20240506           Rome    Clay    FALSE             Bernarda Pera
6989 20240701      Wimbledon   Grass     TRUE               Sofia Kenin
     player_B_name player_A_elo_before player_B_elo_before player_A_win
6674   Iga Swiatek            1421.345            2054.751            0
7138   Iga Swiatek            1489.090            2031.947            0
6946   Iga Swiatek            1491.675            2029.219            0
7455   Iga Swiatek            1460.815            1996.813            0
6513   Iga Swiatek            1497.520            2031.684            0
6989   Iga Swiatek            1510.037            2030.519      

In [20]:
%%R 

# Create confusion matrix
conf_mat <- table(df$predict_R, df$player_A_win)
print(conf_mat) 

# Extract TP, FP, FN
TP <- conf_mat[2,2]
FP <- conf_mat[2,1]
FN <- conf_mat[1,2]
# Calculate Precision and Recall
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
# Print
cat("Precision:", precision, "\n")
cat("Recall:", recall, "\n")

   
       0    1
  0 2520 1149
  1 1502 2759
Precision: 0.6475006 
Recall: 0.7059877 


In [21]:
%%R
write.csv(df, "wta-output.csv", row.names = FALSE)